# 第七章：构建你的智能体框架 HelloAgents

本 Notebook 是第七章 Markdown 文档的交互式版本，让每个代码块都可以直接运行并观察输出结果。

## 本章学习路线
| 节 | 内容 |
|:--|:--|
| 7.1 | 为什么要自建框架 + 快速体验 HelloAgents |
| 7.2 | 扩展 LLM：多供应商 / 本地模型 / 自动检测 |
| 7.3 | 核心接口：Message / Config / Agent 基类 |
| 7.4 | 四种 Agent 范式框架化实现 |
| 7.5 | 工具系统：基类 / 自定义工具 / 工具链 |

> 💡 **学习建议**：先直接运行代码体验效果，再对照 Markdown 文档深入理解设计思路。

---
## 环境准备

运行以下单元格安装所需依赖，并配置 API 密钥。

**API 密钥配置**：在 `code/chapter7/` 目录下新建 `.env` 文件（可参考 `.env.example`），填入你的 LLM API 密钥，格式如下：
```
MODELSCOPE_API_KEY=your-key-here
LLM_MODEL_ID=Qwen/Qwen2.5-72B-Instruct
```

In [1]:
# 安装 HelloAgents 框架（带搜索工具扩展）
# python 版本需要 >= 3.10
import subprocess, sys

print("正在安装 hello-agents...")
subprocess.run([sys.executable, "-m", "pip", "install", "hello-agents==0.1.1", "-q"], check=True)

print("正在安装 python-dotenv...")
subprocess.run([sys.executable, "-m", "pip", "install", "python-dotenv", "-q"], check=True)

print("\n✅ 安装完成！")

正在安装 hello-agents...
正在安装 python-dotenv...

✅ 安装完成！


In [3]:
import os
from dotenv import load_dotenv

load_dotenv()  # 从当前目录 .env 加载
# 验证密钥是否加载成功
key = os.getenv("MODELSCOPE_API_KEY") or os.getenv("OPENAI_API_KEY") or os.getenv("LLM_API_KEY")
if key:
    print(f"✅ 检测到 API 密钥（已隐藏）: {'*' * 8 + key[-4:]}")
else:
    print("⚠️  未检测到 API 密钥，后续涉及 LLM 调用的单元格将无法运行")

✅ 检测到 API 密钥（已隐藏）: ********XJYQ


---
## 7.1 30 秒快速体验 HelloAgents

在理解框架设计之前，先直接体验一下完整效果。

In [5]:
from hello_agents import SimpleAgent, HelloAgentsLLM

# 创建 LLM 实例 —— 框架会自动根据环境变量检测 provider
llm = HelloAgentsLLM()
print(f"🤖 LLM 已初始化，Provider: {llm.provider}，Model: {llm.model}")

# 创建最简单的 SimpleAgent
agent = SimpleAgent(
    name="AI助手",
    llm=llm,
    system_prompt="你是一个有用的AI助手，请用简洁的中文回答"
)

# 基础对话
response = agent.run("你好！请用一句话介绍一下自己")
print(f"\nAgent 回复: {response}")

# 查看对话历史
print(f"\n历史消息数: {len(agent.get_history())} 条")

🤖 LLM 已初始化，Provider: auto，Model: gpt-4.1

Agent 回复: 你好，我是一个智能助手，可以帮助你解答问题、提供信息和协助处理各种任务。

历史消息数: 2 条


---
## 7.2 HelloAgentsLLM 扩展

### 7.2.1 继承方式支持新供应商（以 ModelScope 为例）

通过继承 `HelloAgentsLLM` 来添加新功能，而不是修改库源码。

In [6]:
import os
from typing import Optional
from openai import OpenAI
from hello_agents import HelloAgentsLLM

class MyLLM(HelloAgentsLLM):
    """
    自定义 LLM 客户端：通过继承为 HelloAgentsLLM 添加 ModelScope 支持。
    重写 __init__，拦截 provider='modelscope' 的情况；
    其他 provider 交由父类处理（super().__init__）。
    """
    def __init__(
        self,
        model: Optional[str] = None,
        api_key: Optional[str] = None,
        base_url: Optional[str] = None,
        provider: Optional[str] = "auto",
        **kwargs
    ):
        if provider == "modelscope":
            print("🔧 使用自定义 ModelScope Provider")
            self.provider = "modelscope"
            self.api_key = api_key or os.getenv("MODELSCOPE_API_KEY")
            self.base_url = base_url or "https://api-inference.modelscope.cn/v1/"

            if not self.api_key:
                raise ValueError("ModelScope API key 未找到，请设置 MODELSCOPE_API_KEY 环境变量")

            self.model = model or os.getenv("LLM_MODEL_ID") or "Qwen/Qwen2.5-VL-72B-Instruct"
            self.temperature = kwargs.get("temperature", 0.7)
            self.max_tokens = kwargs.get("max_tokens")
            self.timeout = kwargs.get("timeout", 60)
            self._client = OpenAI(
                api_key=self.api_key,
                base_url=self.base_url,
                timeout=self.timeout
            )
        else:
            # 其他 provider，完全交给父类
            super().__init__(model=model, api_key=api_key, base_url=base_url, provider=provider, **kwargs)


# ── 演示：实例化（无需真正调用 API）──
# 若环境中有 MODELSCOPE_API_KEY，则使用 modelscope；否则回退到自动检测
if os.getenv("MODELSCOPE_API_KEY"):
    my_llm = MyLLM(provider="modelscope")
    print(f"✅ MyLLM 初始化成功，provider={my_llm.provider}, model={my_llm.model}")
else:
    my_llm = MyLLM()  # 自动检测
    print(f"✅ MyLLM 自动检测，provider={my_llm.provider}, model={my_llm.model}")

✅ MyLLM 自动检测，provider=auto, model=gpt-4.1


### 7.2.2 本地模型调用（VLLM / Ollama）

只需改变 `base_url`，其他代码完全不变。

In [5]:
# 本单元格演示配置方式，不实际发起远程调用
from hello_agents import HelloAgentsLLM

print("=== 连接本地 VLLM 服务的配置方式 ===")
print("""
llm_vllm = HelloAgentsLLM(
    provider="vllm",
    model="Qwen/Qwen1.5-0.5B-Chat",
    base_url="http://localhost:8000/v1",
    api_key="vllm"            # 本地服务填任意非空字符串
)
""")

print("=== 连接本地 Ollama 服务的配置方式 ===")
print("""
llm_ollama = HelloAgentsLLM(
    provider="ollama",
    model="llama3",
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)
""")

print("=== 或者通过 .env 文件让框架自动检测 ===")
print("""
# .env 文件内容：
# LLM_BASE_URL=http://localhost:11434/v1
# LLM_MODEL_ID=llama3

llm = HelloAgentsLLM()  # 框架自动检测为 ollama
""")

=== 连接本地 VLLM 服务的配置方式 ===

llm_vllm = HelloAgentsLLM(
    provider="vllm",
    model="Qwen/Qwen1.5-0.5B-Chat",
    base_url="http://localhost:8000/v1",
    api_key="vllm"            # 本地服务填任意非空字符串
)

=== 连接本地 Ollama 服务的配置方式 ===

llm_ollama = HelloAgentsLLM(
    provider="ollama",
    model="llama3",
    base_url="http://localhost:11434/v1",
    api_key="ollama"
)

=== 或者通过 .env 文件让框架自动检测 ===

# .env 文件内容：
# LLM_BASE_URL=http://localhost:11434/v1
# LLM_MODEL_ID=llama3

llm = HelloAgentsLLM()  # 框架自动检测为 ollama



### 7.2.3 自动检测机制

框架按照以下优先级推断 provider：
1. 特定供应商的环境变量（`MODELSCOPE_API_KEY`、`OPENAI_API_KEY` 等）
2. `LLM_BASE_URL` 中的域名/端口特征（`:11434` → ollama，`:8000` → vllm）
3. API 密钥格式辅助判断（`ms-` 前缀 → modelscope）

In [6]:
import re

def simulate_auto_detect(api_key=None, base_url=None, env_overrides=None):
    """
    模拟框架的自动检测逻辑（_auto_detect_provider），
    不实际调用 LLM，仅演示检测规则。
    """
    env = env_overrides or {}

    # 1. 特定供应商环境变量
    if env.get("MODELSCOPE_API_KEY"): return "modelscope"
    if env.get("OPENAI_API_KEY"):     return "openai"
    if env.get("ZHIPU_API_KEY"):      return "zhipu"

    actual_key = api_key or env.get("LLM_API_KEY")
    actual_url = base_url or env.get("LLM_BASE_URL", "")

    # 2. base_url 特征判断
    url_lower = actual_url.lower()
    if "api-inference.modelscope.cn" in url_lower: return "modelscope"
    if "open.bigmodel.cn" in url_lower:            return "zhipu"
    if "localhost" in url_lower or "127.0.0.1" in url_lower:
        if ":11434" in url_lower: return "ollama"
        if ":8000"  in url_lower: return "vllm"
        return "local"

    # 3. API 密钥格式
    if actual_key and actual_key.startswith("ms-"): return "modelscope"

    return "auto"


# ── 多种场景演示 ──
scenarios = [
    {"desc": "设置了 MODELSCOPE_API_KEY",   "env": {"MODELSCOPE_API_KEY": "test-key"}},
    {"desc": "设置了 OPENAI_API_KEY",        "env": {"OPENAI_API_KEY": "sk-test"}},
    {"desc": "base_url 指向本地 Ollama",     "env": {"LLM_BASE_URL": "http://localhost:11434/v1"}},
    {"desc": "base_url 指向本地 VLLM",       "env": {"LLM_BASE_URL": "http://localhost:8000/v1"}},
    {"desc": "API Key 以 ms- 开头",          "api_key": "ms-abcdefg"},
    {"desc": "无任何线索",                    "env": {}},
]

print("自动检测场景演示：")
print("-" * 50)
for s in scenarios:
    provider = simulate_auto_detect(
        api_key=s.get("api_key"),
        base_url=s.get("base_url"),
        env_overrides=s.get("env", {})
    )
    print(f"🔍 {s['desc']:<30} → 检测结果: {provider}")

自动检测场景演示：
--------------------------------------------------
🔍 设置了 MODELSCOPE_API_KEY         → 检测结果: modelscope
🔍 设置了 OPENAI_API_KEY             → 检测结果: openai
🔍 base_url 指向本地 Ollama           → 检测结果: ollama
🔍 base_url 指向本地 VLLM             → 检测结果: vllm
🔍 API Key 以 ms- 开头               → 检测结果: modelscope
🔍 无任何线索                          → 检测结果: auto


---
## 7.3 框架核心接口实现

### 7.3.1 Message 类

用 Pydantic BaseModel 实现，`role` 字段只允许四种取值。

In [7]:
from typing import Optional, Dict, Any, Literal
from datetime import datetime
from pydantic import BaseModel, ValidationError

# 限制 role 只能是这四种值
MessageRole = Literal["user", "assistant", "system", "tool"]

class Message(BaseModel):
    """消息类 —— 对应 hello_agents/core/message.py"""
    content: str
    role: MessageRole
    timestamp: Optional[datetime] = None
    metadata: Optional[Dict[str, Any]] = None

    def __init__(self, content: str, role: MessageRole, **kwargs):
        super().__init__(
            content=content,
            role=role,
            timestamp=kwargs.get("timestamp", datetime.now()),
            metadata=kwargs.get("metadata", {})
        )

    def to_dict(self) -> Dict[str, Any]:
        """转换为 OpenAI API 格式的字典"""
        return {"role": self.role, "content": self.content}

    def __str__(self) -> str:
        return f"[{self.role}] {self.content}"


# ── 演示 ──
msg1 = Message("你好！", "user")
msg2 = Message("你好！我是 AI 助手。", "assistant")
print(f"✅ 消息1: {msg1}")
print(f"✅ 消息2: {msg2}")
print(f"\nOpenAI 格式: {msg1.to_dict()}")
print(f"时间戳: {msg1.timestamp}")

# role 校验：传入非法值会报错
print("\n--- 测试校验 ---")
try:
    bad_msg = Message("测试", "hacker")   # ❌ 非法 role
except ValidationError as e:
    print(f"❌ 校验失败（符合预期）: {e.errors()[0]['msg']}")

✅ 消息1: [user] 你好！
✅ 消息2: [assistant] 你好！我是 AI 助手。

OpenAI 格式: {'role': 'user', 'content': '你好！'}
时间戳: 2026-02-19 14:27:05.389134

--- 测试校验 ---
❌ 校验失败（符合预期）: Input should be 'user', 'assistant', 'system' or 'tool'


### 7.3.2 Config 类

集中管理所有配置，支持从环境变量加载。

In [ ]:
import os
from typing import Optional, Dict, Any
from pydantic import BaseModel

class Config(BaseModel):
    """HelloAgents 配置类 —— 对应 hello_agents/core/config.py"""

    # LLM 配置
    default_model: str = "gpt-3.5-turbo"
    default_provider: str = "openai"
    temperature: float = 0.7
    max_tokens: Optional[int] = None

    # 系统配置
    debug: bool = False
    log_level: str = "INFO"

    # 其他配置
    max_history_length: int = 100

    @classmethod # 声明这是一个类方法，可以直接通过 Config.from_env() 调用，无需实例化类
    def from_env(cls) -> "Config":
        """从环境变量创建配置（代码无需修改，改 .env 即可调整行为）"""
        # cls 代表类本身（即 Config）。
        # 这里调用 cls(...) 等同于直接调用 Config(...) 构造函数，
        # 这种写法的好处是如果子类继承了 Config，cls() 会自动指向子类的构造函数。
        return cls(
            debug=os.getenv("DEBUG", "false").lower() == "true",
            log_level=os.getenv("LOG_LEVEL", "INFO"),
            temperature=float(os.getenv("TEMPERATURE", "0.7")),
            max_tokens=int(os.getenv("MAX_TOKENS")) if os.getenv("MAX_TOKENS") else None,
        )

    def to_dict(self) -> Dict[str, Any]:
        """
        将 Pydantic 模型对象转换为底层的 Python 字典 (dict)。
        它会抓取类中定义的字段（如 default_model, temperature 等）及其当前值，
        导出一个简单的键值对集合，方便后续进行 print 或传参给其他函数。
        """
        return self.model_dump()


# ── 演示 ──
default_config = Config()
print("默认配置 (通过 to_dict() 导出):")
# 这里的 config_dict 就是 {'default_model': 'gpt-3.5-turbo', 'temperature': 0.7, ...}
config_dict = default_config.to_dict()
for k, v in config_dict.items():
    print(f"  {k}: {v}")

# 从环境变量加载
env_config = Config.from_env()
print(f"\n从环境变量加载: debug={env_config.debug}, temperature={env_config.temperature}")

# 覆盖单个配置项
custom_config = Config(temperature=0.9, debug=True, max_tokens=2000)
print(f"\n自定义配置: temperature={custom_config.temperature}, debug={custom_config.debug}, max_tokens={custom_config.max_tokens}")


默认配置:
  default_model: gpt-3.5-turbo
  default_provider: openai
  temperature: 0.7
  max_tokens: None
  debug: False
  log_level: INFO
  max_history_length: 100

从环境变量加载: debug=False, temperature=0.7

自定义配置: temperature=0.9, debug=True, max_tokens=2000


### 7.3.3 Agent 抽象基类

用 ABC + @abstractmethod 强制所有子类必须实现 `run` 方法。

In [9]:
from abc import ABC, abstractmethod
from typing import Optional, Any

# 重用上方定义的 Message、Config（如果重启了 kernel，需先运行上方 cell）

class AgentBase(ABC):
    """Agent 基类 —— 对应 hello_agents/core/agent.py"""

    def __init__(self, name: str, system_prompt: Optional[str] = None):
        self.name = name
        self.system_prompt = system_prompt
        self._history: list = []   # 存放 Message 对象
        print(f"🤖 Agent '{name}' 已创建")

    @abstractmethod
    def run(self, input_text: str, **kwargs) -> str:
        """子类必须实现此方法 —— 这是 Agent 的唯一对外接口"""
        pass

    # ── 通用工具方法（子类继承，无需重写）──
    def add_message(self, content: str, role: str):
        self._history.append({"role": role, "content": content})

    def clear_history(self):
        self._history.clear()

    def get_history(self) -> list:
        return self._history.copy()

    def __str__(self) -> str:
        return f"Agent(name={self.name})"


# ── 验证：抽象类不能直接实例化 ──
print("--- 尝试直接实例化抽象类 ---")
try:
    a = AgentBase("测试")
except TypeError as e:
    print(f"❌ 符合预期：{e}")

# ── 验证：子类必须实现 run ──
print("\n--- 子类未实现 run ---")
try:
    class LazyAgent(AgentBase):
        pass   # 没有实现 run()
    LazyAgent("懒鬼")
except TypeError as e:
    print(f"❌ 符合预期：{e}")

# ── 正确实现 ──
print("\n--- 正确实现子类 ---")
class EchoAgent(AgentBase):
    def run(self, input_text: str, **kwargs) -> str:
        response = f"你说了：{input_text}"
        self.add_message(input_text, "user")
        self.add_message(response, "assistant")
        return response

echo = EchoAgent("回声机器人")
print(echo.run("你好世界"))
print(f"历史记录: {echo.get_history()}")

--- 尝试直接实例化抽象类 ---
❌ 符合预期：Can't instantiate abstract class AgentBase without an implementation for abstract method 'run'

--- 子类未实现 run ---
❌ 符合预期：Can't instantiate abstract class LazyAgent without an implementation for abstract method 'run'

--- 正确实现子类 ---
🤖 Agent '回声机器人' 已创建
你说了：你好世界
历史记录: [{'role': 'user', 'content': '你好世界'}, {'role': 'assistant', 'content': '你说了：你好世界'}]


---
## 7.4 四种 Agent 范式框架化实现

### 7.4.1 MySimpleAgent

继承框架的 `SimpleAgent`，添加可选工具调用 + 流式响应功能。

In [8]:
import re
from typing import Optional, Iterator
from hello_agents import SimpleAgent, HelloAgentsLLM, Config, Message

class MySimpleAgent(SimpleAgent):
    """
    重写的简单对话 Agent。
    相比基类新增：
      - 可选工具调用（tool_registry + enable_tool_calling）
      - 流式响应（stream_run）
      - 动态工具管理（add_tool / list_tools / has_tools）
    """

    def __init__(
        self,
        name: str,
        llm: HelloAgentsLLM,
        system_prompt: Optional[str] = None,
        config: Optional[Config] = None,
        tool_registry=None,
        enable_tool_calling: bool = True
    ):
        super().__init__(name, llm, system_prompt, config)
        self.tool_registry = tool_registry
        # 只有同时有注册表且用户允许时，才真正启用工具
        self.enable_tool_calling = enable_tool_calling and tool_registry is not None
        print(f"✅ {name} 初始化完成，工具调用: {'启用' if self.enable_tool_calling else '禁用'}")

    # ── 核心方法：run ──
    def run(self, input_text: str, max_tool_iterations: int = 3, **kwargs) -> str:
        print(f"\n🤖 {self.name} 正在处理: {input_text}")
        messages = []

        # 系统提示词（含工具说明）
        messages.append({"role": "system", "content": self._get_enhanced_system_prompt()})

        # 加入历史消息
        for msg in self._history:
            messages.append({"role": msg.role, "content": msg.content})

        messages.append({"role": "user", "content": input_text})

        if not self.enable_tool_calling:
            response = self.llm.invoke(messages, **kwargs)
            # 这里的两行是将本次对话存入“记忆”库中
            self.add_message(Message(input_text, "user")) # 记录用户说的话
            self.add_message(Message(response, "assistant")) # 记录 AI 的回答
            print(f"✅ {self.name} 响应完成")
            return response

        return self._run_with_tools(messages, input_text, max_tool_iterations, **kwargs)

    def _get_enhanced_system_prompt(self) -> str:
        base = self.system_prompt or "你是一个有用的AI助手。"
        if not self.enable_tool_calling or not self.tool_registry:
            return base
        tools_desc = self.tool_registry.get_tools_description()
        if not tools_desc or tools_desc == "暂无可用工具":
            return base
        return (
            base
            + "\n\n## 可用工具\n" + tools_desc
            + "\n\n## 工具调用格式\n"
            + "`[TOOL_CALL:{tool_name}:{parameters}]`\n"
            + "例如: `[TOOL_CALL:calculator:15*8+32]`\n"
        )

    def _run_with_tools(self, messages, input_text, max_iterations, **kwargs):
        iteration, final_response = 0, ""
        while iteration < max_iterations:
            response = self.llm.invoke(messages, **kwargs)
            calls = self._parse_tool_calls(response)
            if calls:
                print(f"🔧 检测到 {len(calls)} 个工具调用")
                clean = response
                results = []
                for c in calls:
                    results.append(self._execute_tool_call(c["tool_name"], c["parameters"]))
                    clean = clean.replace(c["original"], "")
                messages.append({"role": "assistant", "content": clean})
                messages.append({"role": "user", "content": "工具结果:\n" + "\n".join(results) + "\n请基于结果作答。"})
                iteration += 1
                continue
            final_response = response
            break
        if iteration >= max_iterations and not final_response:
            final_response = self.llm.invoke(messages, **kwargs)
        
        self.add_message(Message(input_text, "user"))
        self.add_message(Message(final_response, "assistant"))
        print(f"✅ {self.name} 响应完成")
        return final_response

    def _parse_tool_calls(self, text: str) -> list:
        pattern = r'\[TOOL_CALL:([^:]+):([^\]]+)\]'
        return [
            {"tool_name": tn.strip(), "parameters": p.strip(),
             "original": f"[TOOL_CALL:{tn}:{p}]"}
            for tn, p in re.findall(pattern, text)
        ]

    def _execute_tool_call(self, tool_name: str, parameters: str) -> str:
        if not self.tool_registry:
            return "❌ 未配置工具注册表"
        try:
            result = self.tool_registry.execute_tool(tool_name, parameters)
            return f"🔧 {tool_name} 结果:\n{result}"
        except Exception as e:
            return f"❌ 工具调用失败: {e}"

    # ── 流式响应 ──
    def stream_run(self, input_text: str, **kwargs) -> Iterator[str]:
        print(f"\n🌊 {self.name} 开始流式处理: {input_text}")
        messages = []
        if self.system_prompt:
            messages.append({"role": "system", "content": self.system_prompt})
        for msg in self._history:
            messages.append({"role": msg.role, "content": msg.content})
        messages.append({"role": "user", "content": input_text})

        full_response = ""
        print("📝 实时响应: ", end="")
        try:
            for chunk in self.llm.stream_invoke(messages, **kwargs):
                full_response += chunk
                print(chunk, end="", flush=True) #flush=True 确保每次输出都立即显示
                yield chunk
        except Exception:
            # 底层库偶发空 chunk 时跳过，已收到的内容保持原样
            pass
        print()
        # 记录流式对话的历史
        self.add_message(Message(input_text, "user"))
        self.add_message(Message(full_response, "assistant"))
        print(f"✅ {self.name} 流式响应完成")

    # ── 工具管理便利方法 ──
    def add_tool(self, tool) -> None:
        if not self.tool_registry:
            from hello_agents import ToolRegistry
            self.tool_registry = ToolRegistry()
            self.enable_tool_calling = True
        self.tool_registry.register_tool(tool)
        print(f"🔧 工具 '{tool.name}' 已添加")

    def has_tools(self) -> bool:
        return self.enable_tool_calling and self.tool_registry is not None

    def list_tools(self) -> list:
        return self.tool_registry.list_tools() if self.tool_registry else []


print("✅ MySimpleAgent 类 definition 完成")


✅ MySimpleAgent 类 definition 完成


In [9]:
# ── 测试 MySimpleAgent ──
from hello_agents import HelloAgentsLLM, ToolRegistry
from hello_agents.tools import CalculatorTool

llm = HelloAgentsLLM()

# 测试1：基础对话（无工具）
print("=" * 40)
print("测试1：基础对话")
basic_agent = MySimpleAgent(
    name="基础助手",
    llm=llm,
    system_prompt="你是一个友好的AI助手，请用简洁的中文回答。"
)
r1 = basic_agent.run("你好，请用一句话介绍一下人工智能")
print(f"回复: {r1}")

# 测试2：带工具的 Agent
print("\n" + "=" * 40)
print("测试2：工具增强对话")
registry = ToolRegistry()
registry.register_tool(CalculatorTool())

enhanced_agent = MySimpleAgent(
    name="计算助手",
    llm=llm,
    system_prompt="你是一个智能助手，可以使用工具来帮助用户。",
    tool_registry=registry,
    enable_tool_calling=True
)
r2 = enhanced_agent.run("请帮我计算 15 * 8 + 32")
print(f"回复: {r2}")

# 测试3：流式响应
print("\n" + "=" * 40)
print("测试3：流式响应")
for _ in basic_agent.stream_run("请用一句话解释什么是机器学习"):
    pass  # 内容已在 stream_run 中实时打印

# 测试4：动态添加工具
print("\n" + "=" * 40)
print("测试4：动态工具管理")
print(f"添加工具前: has_tools={basic_agent.has_tools()}")
basic_agent.add_tool(CalculatorTool())
print(f"添加工具后: has_tools={basic_agent.has_tools()}, 工具列表={basic_agent.list_tools()}")
print(f"\n对话历史共 {len(basic_agent.get_history())} 条")

测试1：基础对话
✅ 基础助手 初始化完成，工具调用: 禁用

🤖 基础助手 正在处理: 你好，请用一句话介绍一下人工智能
✅ 基础助手 响应完成
回复: 人工智能是让机器具备模拟人类思维和行为能力的技术。

测试2：工具增强对话
✅ 工具 'python_calculator' 已注册。
✅ 计算助手 初始化完成，工具调用: 启用

🤖 计算助手 正在处理: 请帮我计算 15 * 8 + 32
🔧 检测到 1 个工具调用
🧮 正在计算: 15*8+32
✅ 计算结果: 152
✅ 计算助手 响应完成
回复: 计算结果是152。

测试3：流式响应

🌊 基础助手 开始流式处理: 请用一句话解释什么是机器学习
📝 实时响应: 🧠 正在调用 gpt-4.1 模型...
✅ 大语言模型响应成功:
❌ 调用LLM API时发生错误: list index out of range

✅ 基础助手 流式响应完成

测试4：动态工具管理
添加工具前: has_tools=False
✅ 工具 'python_calculator' 已注册。
🔧 工具 'python_calculator' 已添加
添加工具后: has_tools=True, 工具列表=['python_calculator']

对话历史共 4 条


### 7.4.2 MyReActAgent

经典的 Thought → Action → Observation 循环，每次只执行一步。

In [12]:
# ReAct 的提示词模板（通用化版本）
MY_REACT_PROMPT = """你是一个具备推理和行动能力的AI助手。

## 可用工具
{tools}

## 工作流程
请严格按照以下格式进行回应，每次只能执行一个步骤:

Thought: 分析当前问题，思考需要什么信息或采取什么行动。
Action: 选择一个行动，格式必须是以下之一:
- `{{tool_name}}[{{tool_input}}]` - 调用指定工具
- `Finish[最终答案]` - 当你有足够信息给出最终答案时

## 重要提醒
1. 每次回应必须包含 Thought 和 Action 两部分
2. 工具调用格式必须是: 工具名[参数]
3. 只有确信有足够信息时，才使用 Finish

## 当前任务
**Question:** {question}

## 执行历史
{history}

现在开始你的推理和行动:
"""

print("✅ ReAct 提示词模板已定义")

✅ ReAct 提示词模板已定义


In [13]:
import re
from typing import Optional, List, Tuple
from hello_agents import ReActAgent, HelloAgentsLLM, Config, Message, ToolRegistry

class MyReActAgent(ReActAgent):
    """重写的 ReAct Agent"""

    def __init__(
        self,
        name: str,
        llm: HelloAgentsLLM,
        tool_registry: ToolRegistry,
        system_prompt: Optional[str] = None,
        config: Optional[Config] = None,
        max_steps: int = 5,
        custom_prompt: Optional[str] = None
    ):
        super().__init__(name, llm, system_prompt, config)
        self.tool_registry = tool_registry
        self.max_steps = max_steps
        self.current_history: List[str] = []
        self.prompt_template = custom_prompt or MY_REACT_PROMPT
        print(f"✅ {name} 初始化完成，最大步数: {max_steps}")

    def run(self, input_text: str, **kwargs) -> str:
        self.current_history = []
        step = 0
        print(f"\n🤖 {self.name} 开始处理: {input_text}")

        while step < self.max_steps:
            step += 1
            print(f"\n--- 第 {step} 步 ---")

            # 1. 构建提示词
            prompt = self.prompt_template.format(
                tools=self.tool_registry.get_tools_description(),
                question=input_text,
                history="\n".join(self.current_history)
            )
            messages = [{"role": "user", "content": prompt}]

            # 2. 调用 LLM
            response = self.llm.invoke(messages, **kwargs)
            print(f"LLM 输出:\n{response}")

            # 3. 解析 thought / action
            thought, action = self._parse_output(response)

            # 4. Finish → 返回答案
            if action and action.startswith("Finish"):
                final = self._parse_action_input(action)
                self.add_message(Message(input_text, "user"))
                self.add_message(Message(final, "assistant"))
                print(f"\n🎉 最终答案: {final}")
                return final

            # 5. 工具调用
            if action:
                tool_name, tool_input = self._parse_action(action)
                obs = self.tool_registry.execute_tool(tool_name, tool_input)
                print(f"\nObservation: {obs}")
                self.current_history.append(f"Action: {action}")
                self.current_history.append(f"Observation: {obs}")

        # 超出最大步数
        ans = "抱歉，无法在限定步数内完成任务。"
        self.add_message(Message(input_text, "user"))
        self.add_message(Message(ans, "assistant"))
        return ans


print("✅ MyReActAgent 类定义完成")

✅ MyReActAgent 类定义完成


In [14]:
from hello_agents import HelloAgentsLLM, ToolRegistry
from hello_agents.tools import CalculatorTool

llm = HelloAgentsLLM()

# 注册工具
tool_registry = ToolRegistry()
tool_registry.register_tool(CalculatorTool())

# 创建 MyReActAgent
react_agent = MyReActAgent(
    name="ReAct助手",
    llm=llm,
    tool_registry=tool_registry,
    max_steps=5
)

# 测试：需要工具的任务
result = react_agent.run("请计算 (25 + 75) * 2 的结果")
print(f"\n=== 最终结果 ===\n{result}")

✅ 工具 'python_calculator' 已注册。
✅ ReAct助手 初始化完成，最大步数: 5

🤖 ReAct助手 开始处理: 请计算 (25 + 75) * 2 的结果

--- 第 1 步 ---
LLM 输出:
Thought: 首先需要计算括号内的内容，然后再将结果乘以2。可以直接进行数学运算。
Action: python_calculator[(25 + 75) * 2]
🧮 正在计算: (25 + 75) * 2
✅ 计算结果: 200

Observation: 200

--- 第 2 步 ---
LLM 输出:
Thought: 我已经得到了计算结果，该表达式的值是200，可以给出最终答案。
Action: Finish[200]

🎉 最终答案: 200

=== 最终结果 ===
200


### 7.4.3 ReflectionAgent 提示词模板

这里提供通用化的提示词模板，你可以参考第四章的实现来补全 `MyReflectionAgent` 的代码。

In [17]:
# ReflectionAgent 默认提示词（通用化版本，适用于写作、分析、代码等多种场景）
DEFAULT_REFLECTION_PROMPTS = {
    "initial": """
请根据以下要求完成任务:

任务: {task}

请提供一个完整、准确的回答。
""",
    "reflect": """
请仔细审查以下回答，并找出可能的问题或改进空间:

# 原始任务:
{task}

# 当前回答:
{content}

请分析这个回答的质量，指出不足之处，并提出具体的改进建议。
如果回答已经很好，请回答
。
""",
    "refine": """
请根据反馈意见改进你的回答:

# 原始任务:
{task}

# 上一轮回答:
{last_attempt}

# 反馈意见:
{feedback}

请提供一个改进后的回答。
"""
}

# 用于代码生成场景的自定义提示词（类似第四章）
CODE_REFLECTION_PROMPTS = {
    "initial": "你是Python专家，请编写函数:\n{task}",
    "reflect": "请审查代码的算法效率:\n任务:{task}\n代码:{content}",
    "refine":  "请根据反馈优化代码:\n任务:{task}\n反馈:{feedback}"
}

print("✅ ReflectionAgent 提示词模板已定义")
print("\n通用提示词包含三个阶段:")
for key in DEFAULT_REFLECTION_PROMPTS:
    print(f"  - {key}: {DEFAULT_REFLECTION_PROMPTS[key].strip()[:60]}...")

✅ ReflectionAgent 提示词模板已定义

通用提示词包含三个阶段:
  - initial: 请根据以下要求完成任务:

任务: {task}

请提供一个完整、准确的回答。...
  - reflect: 请仔细审查以下回答，并找出可能的问题或改进空间:

# 原始任务:
{task}

# 当前回答:
{content}
...
  - refine: 请根据反馈意见改进你的回答:

# 原始任务:
{task}

# 上一轮回答:
{last_attempt}

# 反...


In [18]:
from hello_agents import ReflectionAgent, HelloAgentsLLM, Config, Message
from typing import Optional, Dict

class MyReflectionAgent(ReflectionAgent):
    """
    重写的 Reflection Agent：执行 → 反思 → 优化 循环。
    支持自定义提示词，可覆盖 initial / reflect / refine 三个阶段。
    """

    def __init__(
        self,
        name: str,
        llm: HelloAgentsLLM,
        system_prompt: Optional[str] = None,
        config: Optional[Config] = None,
        max_reflections: int = 2,
        custom_prompts: Optional[Dict[str, str]] = None
    ):
        super().__init__(name, llm, system_prompt, config)
        self.max_reflections = max_reflections
        # 使用自定义提示词，否则用默认
        self.prompts = custom_prompts or DEFAULT_REFLECTION_PROMPTS
        print(f"✅ {name} 初始化完成，最大反思轮数: {max_reflections}")

    def run(self, input_text: str, **kwargs) -> str:
        print(f"\n🤔 {self.name} 开始处理: {input_text}")

        # 第一轮：生成初稿
        initial_prompt = self.prompts["initial"].format(task=input_text)
        current = self.llm.invoke(
            [{"role": "user", "content": initial_prompt}], **kwargs
        )
        print(f"\n📝 初稿 (前100字): {current[:100]}...")

        # 反思-优化循环
        for i in range(self.max_reflections):
            reflect_prompt = self.prompts["reflect"].format(
                task=input_text, content=current
            )
            feedback = self.llm.invoke(
                [{"role": "user", "content": reflect_prompt}], **kwargs
            )
            print(f"\n🔍 第{i+1}次反思: {feedback[:80]}...")

            if "无需改进" in feedback:
                print("✨ 质量已满足要求，提前结束反思")
                break

            refine_prompt = self.prompts["refine"].format(
                task=input_text, last_attempt=current, feedback=feedback
            )
            current = self.llm.invoke(
                [{"role": "user", "content": refine_prompt}], **kwargs
            )
            print(f"✅ 第{i+1}轮优化完成")

        self.add_message(Message(input_text, "user"))
        self.add_message(Message(current, "assistant"))
        return current


print("✅ MyReflectionAgent 类定义完成")

✅ MyReflectionAgent 类定义完成


In [19]:
# 测试 MyReflectionAgent
from hello_agents import HelloAgentsLLM

llm = HelloAgentsLLM()

# 通用写作场景
general_agent = MyReflectionAgent(name="通用反思助手", llm=llm, max_reflections=2)
result = general_agent.run("写一句话介绍机器学习")
print(f"\n=== 最终结果 ===\n{result}")

✅ 通用反思助手 初始化完成，最大反思轮数: 2

🤔 通用反思助手 开始处理: 写一句话介绍机器学习

📝 初稿 (前100字): 机器学习是一种让计算机通过数据自动学习并改进自身性能的人工智能技术。...

🔍 第1次反思: 这句话总体来说简明扼要，能够基本表达机器学习的核心思想，即“通过数据自动学习并改进自身性能”。但仍有一些细节可以改进：

**不足之处：**
1. “改进自身性...
✅ 第1轮优化完成

🔍 第2次反思: 这条回答总体来说表达清晰，涵盖了机器学习的基本定义、核心方法（算法分析数据、自动识别规律）、以及应用（预测、分类、决策），也指出了机器学习与传统明确编程的区别。...
✅ 第2轮优化完成

=== 最终结果 ===
机器学习是一种人工智能方法，它让计算机通过分析数据自动学习规律，无需人工指定具体规则，从而实现预测、分类等任务，例如识别图片中的物体。


### 7.4.4 PlanAndSolveAgent 提示词模板

框架化版本强制 Planner 用 Python 列表格式输出计划，保证 Executor 能稳定解析。

In [20]:
DEFAULT_PLANNER_PROMPT = """
你是一个顶级的AI规划专家。请将用户的复杂问题分解成一个由多个简单步骤组成的行动计划。
确保每个步骤都是独立的、可执行的子任务，并严格按照逻辑顺序排列。
你的输出必须是一个 Python 列表，其中每个元素都是一个描述子任务的字符串。

问题: {question}

请严格按照以下格式输出你的计划:
```python
["步骤1", "步骤2", "步骤3", ...]
```
"""

DEFAULT_EXECUTOR_PROMPT = """
你是一位顶级的AI执行专家。请严格按照给定的计划，一步步地解决问题。
你将收到原始问题、完整的计划，以及到目前为止已经完成的步骤和结果。
请你专注于解决
，仅输出该步骤的最终答案，不要输出额外的解释。

# 原始问题:
{question}

# 完整计划:
{plan}

# 历史步骤与结果:
{history}

# 当前步骤:
{current_step}

请仅输出针对
的回答:
"""

print("✅ PlanAndSolve 提示词模板已定义")

✅ PlanAndSolve 提示词模板已定义


In [22]:
import re
import ast
from typing import Optional, Dict, List
from hello_agents import PlanAndSolveAgent, HelloAgentsLLM, Config, Message

class MyPlanAndSolveAgent(PlanAndSolveAgent):
    """重写的 Plan-and-Solve Agent"""

    def __init__(
        self,
        name: str,
        llm: HelloAgentsLLM,
        system_prompt: Optional[str] = None,
        config: Optional[Config] = None,
        custom_prompts: Optional[Dict[str, str]] = None
    ):
        super().__init__(name, llm, system_prompt, config)
        prompts = custom_prompts or {}
        self.planner_prompt = prompts.get("planner", DEFAULT_PLANNER_PROMPT)
        self.executor_prompt = prompts.get("executor", DEFAULT_EXECUTOR_PROMPT)
        print(f"✅ {name} 初始化完成")

    def run(self, input_text: str, **kwargs) -> str:
        print(f"\n📋 {self.name} 开始规划: {input_text}")

        # ── 第一步：Planner 生成计划 ──
        plan_response = self.llm.invoke(
            [{"role": "user", "content": self.planner_prompt.format(question=input_text)}],
            **kwargs
        )
        plan = self._parse_plan(plan_response)
        if not plan:
            return "❌ 规划失败，无法解析计划"

        print(f"\n📝 规划结果 ({len(plan)} 步):")
        for i, step in enumerate(plan, 1):
            print(f"  {i}. {step}")

        # ── 第二步：Executor 逐步执行 ──
        history: List[str] = []
        final_result = ""

        for i, step in enumerate(plan, 1):
            print(f"\n🔄 执行步骤 {i}/{len(plan)}: {step}")
            exec_prompt = self.executor_prompt.format(
                question=input_text,
                plan="\n".join(f"{j+1}. {s}" for j, s in enumerate(plan)),
                history="\n".join(history) or "（暂无）",
                current_step=step
            )
            result = self.llm.invoke([{"role": "user", "content": exec_prompt}], **kwargs)
            history.append(f"步骤{i}({step}): {result}")
            final_result = result
            print(f"   结果: {result[:80]}..." if len(result) > 80 else f"   结果: {result}")

        self.add_message(Message(input_text, "user"))
        self.add_message(Message(final_result, "assistant"))
        return final_result

    def _parse_plan(self, text: str) -> List[str]:
        """从 LLM 输出中解析 Python 列表格式的计划"""
        match = re.search(r'```python\s*([\s\S]+?)```', text)
        code = match.group(1).strip() if match else text.strip()
        list_match = re.search(r'\[.+\]', code, re.DOTALL)
        if list_match:
            try:
                return ast.literal_eval(list_match.group())
            except Exception:
                pass
        return []


print("✅ MyPlanAndSolveAgent 类定义完成")

✅ MyPlanAndSolveAgent 类定义完成


In [23]:
from hello_agents import HelloAgentsLLM

llm = HelloAgentsLLM()
agent = MyPlanAndSolveAgent(name="规划执行助手", llm=llm)

question = "一个水果店周一卖出了15个苹果。周二卖出的苹果数量是周一的两倍。" \
           "周三卖出的数量比周二少了5个。请问这三天总共卖出了多少个苹果？"

result = agent.run(question)
print(f"\n=== 最终结果 ===\n{result}")

✅ 规划执行助手 初始化完成

📋 规划执行助手 开始规划: 一个水果店周一卖出了15个苹果。周二卖出的苹果数量是周一的两倍。周三卖出的数量比周二少了5个。请问这三天总共卖出了多少个苹果？

📝 规划结果 (4 步):
  1. 确定周一卖出的苹果数量为15个
  2. 计算周二卖出的苹果数量，即周一的两倍
  3. 计算周三卖出的苹果数量，即周二的数量减去5个
  4. 将周一、周二、周三卖出的苹果数量相加，得到三天的总销售数量

🔄 执行步骤 1/4: 确定周一卖出的苹果数量为15个
   结果: 15

🔄 执行步骤 2/4: 计算周二卖出的苹果数量，即周一的两倍
   结果: 30

🔄 执行步骤 3/4: 计算周三卖出的苹果数量，即周二的数量减去5个
   结果: 25

🔄 执行步骤 4/4: 将周一、周二、周三卖出的苹果数量相加，得到三天的总销售数量
   结果: 70

=== 最终结果 ===
70


---
## 7.5 工具系统

### 7.5.1 Tool 基类 + ToolRegistry 注册机制

In [23]:
from abc import ABC, abstractmethod
from typing import Dict, Any, List, Callable, Optional
from pydantic import BaseModel

# ── ToolParameter：描述一个工具参数 ──
class ToolParameter(BaseModel):
    name: str
    type: str
    description: str
    required: bool = True
    default: Any = None

# ── Tool 基类 ──
class Tool(ABC):
    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description

    @abstractmethod
    def run(self, parameters: Dict[str, Any]) -> str:
        """执行工具 —— 子类必须实现"""
        pass

    @abstractmethod
    def get_parameters(self) -> List[ToolParameter]:
        """获取参数定义 —— 子类必须实现"""
        pass


# ── 一个最小化的 SimpleToolRegistry（演示用） ──
class SimpleToolRegistry:
    def __init__(self):
        self._tools: Dict[str, Tool] = {}
        self._functions: Dict[str, dict] = {}

    def register_tool(self, tool: Tool):
        if tool.name in self._tools:
            print(f"⚠️  工具 '{tool.name}' 已存在，将被覆盖")
        self._tools[tool.name] = tool
        print(f"✅ 工具 '{tool.name}' 已注册")

    def register_function(self, name: str, description: str, func: Callable[[str], str]):
        self._functions[name] = {"description": description, "func": func}
        print(f"✅ 函数工具 '{name}' 已注册")

    def get_tools_description(self) -> str:
        lines = [f"- {t.name}: {t.description}" for t in self._tools.values()]
        lines += [f"- {n}: {i['description']}" for n, i in self._functions.items()]
        return "\n".join(lines) or "暂无可用工具"

    def execute_tool(self, name: str, input_data: str) -> str:
        if name in self._tools:
            return self._tools[name].run({"input": input_data})
        if name in self._functions:
            return self._functions[name]["func"](input_data)
        return f"❌ 未找到工具: {name}"


# ── 演示 ──
print("=== 工具基础设施演示 ===")

# 1. 用函数方式快速注册
registry = SimpleToolRegistry()

def simple_upper(text: str) -> str:
    return text.upper()

registry.register_function(
    name="upper",
    description="将文本转为大写",
    func=simple_upper
)

print(f"\n工具描述:\n{registry.get_tools_description()}")
print(f"\n执行结果: {registry.execute_tool('upper', 'hello world')}")

=== 工具基础设施演示 ===
✅ 函数工具 'upper' 已注册

工具描述:
- upper: 将文本转为大写

执行结果: HELLO WORLD


### 7.5.2 自定义计算工具

In [24]:
import ast
import operator
import math
from hello_agents import ToolRegistry

def my_calculate(expression: str) -> str:
    """安全的数学计算，支持 +、-、*、/、sqrt"""
    if not expression.strip():
        return "计算表达式不能为空"

    _operators = {
        ast.Add:  operator.add,
        ast.Sub:  operator.sub,
        ast.Mult: operator.mul,
        ast.Div:  operator.truediv,
    }
    _functions = {"sqrt": math.sqrt, "pi": math.pi}

    def _eval(node):
        if isinstance(node, ast.Constant):   return node.value
        if isinstance(node, ast.BinOp):
            op = _operators.get(type(node.op))
            return op(_eval(node.left), _eval(node.right))
        if isinstance(node, ast.Call):
            fn = node.func.id
            if fn in _functions:
                return _functions[fn](*[_eval(a) for a in node.args])
        if isinstance(node, ast.Name) and node.id in _functions:
            return _functions[node.id]
        raise ValueError(f"不支持的表达式: {ast.dump(node)}")

    try:
        result = _eval(ast.parse(expression, mode="eval").body)
        return str(result)
    except Exception as e:
        return f"计算失败: {e}"


# 注册到 ToolRegistry
registry = ToolRegistry()
registry.register_function(
    name="my_calculator",
    description="安全的数学计算工具，支持 +、-、*、/、sqrt(x)",
    func=my_calculate
)

# ── 测试 ──
print("=== 计算器工具测试 ===")
test_cases = [
    "2 + 3",
    "10 - 4",
    "5 * 6",
    "15 / 3",
    "sqrt(16)",
    "2 + 3 * 4",    # 运算符优先级
    "sqrt(16) + 2 * 3",
    "10 / 0",        # 除零错误
]

for expr in test_cases:
    result = registry.execute_tool("my_calculator", expr)
    print(f"  {expr:<25} → {result}")

✅ 工具 'my_calculator' 已注册。
=== 计算器工具测试 ===
  2 + 3                     → 5
  10 - 4                    → 6
  5 * 6                     → 30
  15 / 3                    → 5.0
  sqrt(16)                  → 4.0
  2 + 3 * 4                 → 14
  sqrt(16) + 2 * 3          → 10.0
  10 / 0                    → 计算失败: division by zero


### 7.5.3 多源搜索工具

整合 Tavily 和 SerpAPI，实现智能降级。

In [24]:
import os
from hello_agents import ToolRegistry

class MyAdvancedSearchTool:
    """多源搜索工具：优先 Tavily，降级到 SerpAPI"""

    def __init__(self):
        self.name = "my_advanced_search"
        self.description = "智能搜索工具，支持多个搜索源，自动选择最佳结果"
        self.search_sources = []
        self.tavily_client = None
        self._setup()

    def _setup(self):
        if os.getenv("TAVILY_API_KEY"):
            try:
                from tavily import TavilyClient
                self.tavily_client = TavilyClient(api_key=os.getenv("TAVILY_API_KEY"))
                self.search_sources.append("tavily")
                print("✅ Tavily 搜索源已启用")
            except ImportError:
                print("⚠️  tavily 库未安装，运行: pip install hello-agents[search]==0.1.1")

        if os.getenv("SERPAPI_API_KEY"):
            try:
                import serpapi  # noqa
                self.search_sources.append("serpapi")
                print("✅ SerpAPI 搜索源已启用")
            except ImportError:
                print("⚠️  serpapi 库未安装")

        if not self.search_sources:
            print("⚠️  无可用搜索源（未配置 TAVILY_API_KEY / SERPAPI_API_KEY）")

    def search(self, query: str) -> str:
        if not query.strip():
            return "❌ 搜索内容不能为空"
        if not self.search_sources:
            return "❌ 没有可用搜索源，请在 .env 中配置 TAVILY_API_KEY 或 SERPAPI_API_KEY"

        for source in self.search_sources:
            try:
                if source == "tavily":   return self._tavily(query)
                if source == "serpapi": return self._serpapi(query)
            except Exception as e:
                print(f"⚠️  {source} 失败: {e}")
        return "❌ 所有搜索源均失败"

    def _tavily(self, query: str) -> str:
        resp = self.tavily_client.search(query=query, max_results=3)
        out = f"💡 Tavily: {resp.get('answer', '')}\n\n"
        for i, item in enumerate(resp.get("results", [])[:3], 1):
            out += f"[{i}] {item.get('title','')}\n    {item.get('content','')[:150]}...\n\n"
        return out

    def _serpapi(self, query: str) -> str:
        import serpapi
        r = serpapi.GoogleSearch({"q": query, "api_key": os.getenv("SERPAPI_API_KEY"), "num": 3}).get_dict()
        out = "🌐 Google 搜索结果:\n"
        for i, res in enumerate(r.get("organic_results", [])[:3], 1):
            out += f"[{i}] {res.get('title','')}\n    {res.get('snippet','')}\n\n"
        return out


def create_advanced_search_registry() -> ToolRegistry:
    registry = ToolRegistry()
    tool = MyAdvancedSearchTool()
    registry.register_function(
        name="advanced_search",
        description=tool.description,
        func=tool.search
    )
    return registry


print("\n=== 搜索工具注册表 ===")
search_registry = create_advanced_search_registry()
print(f"\n工具描述:\n{search_registry.get_tools_description()}")

# 测试搜索（需配置 API Key）
if os.getenv("TAVILY_API_KEY") or os.getenv("SERPAPI_API_KEY"):
    print("\n=== 搜索测试 ===")
    result = search_registry.execute_tool("advanced_search", "Python 编程语言的历史")
    print(result[:500])
else:
    print("\n⚠️  未配置搜索 API Key，跳过搜索测试")
    print("   配置方式：在 .env 文件中添加 TAVILY_API_KEY=your-key")


=== 搜索工具注册表 ===
✅ Tavily 搜索源已启用
✅ SerpAPI 搜索源已启用
✅ 工具 'advanced_search' 已注册。

工具描述:
- advanced_search: 智能搜索工具，支持多个搜索源，自动选择最佳结果

=== 搜索测试 ===
💡 Tavily: None

[1] Python编程语言发展简史 - 腾讯云
    Python由荷兰人Guido von Rossum于1989年圣诞节期间创立，受ABC语言启发，旨在创造易学易用且功能全面的编程语言。Python结合C语言性能与shell脚本便捷...

[2] Python历史- Python教程- 廖雪峰的官方网站
    ### Python教程. 使用\_\_slots\_\_. # Python历史. Python是著名的“龟叔”Guido van Rossum在1989年圣诞节期间，为了打发无聊的圣诞节而编写的一个编程语言。. 现在，全世界差不多有600多种编程语言，但流行的编程语言也就那么20来种。如果你听说...

[3] Python的前世今生：一门语言的崛起与演变 - 知乎专栏
    Python的前世可以追溯到上世纪80年代末，它的创造者是荷兰程序员Guido van Rossum。Guido的目标是创建一门简单、易读且高效的编程语言，以解决他在Amoeba...




### 7.5.4 工具链与异步执行

In [25]:
from typing import Dict, Any, List, Optional
from hello_agents import ToolRegistry

class ToolChain:
    """工具链：将多个工具按顺序串联，前一步的输出可传入后一步"""

    def __init__(self, name: str, description: str):
        self.name = name
        self.description = description
        self.steps: List[Dict[str, Any]] = []

    def add_step(self, tool_name: str, input_template: str, output_key: str = None):
        """
        添加一个步骤。
        input_template: 支持 {input}、{step_0_result} 等变量占位符
        output_key: 本步骤结果存入 context 的键名
        """
        self.steps.append({
            "tool_name": tool_name,
            "input_template": input_template,
            "output_key": output_key or f"step_{len(self.steps)}_result"
        })

    def execute(self, registry: ToolRegistry, initial_input: str,
                context: Optional[Dict[str, Any]] = None) -> str:
        ctx = context or {}
        ctx["input"] = initial_input
        print(f"🔗 开始执行工具链: {self.name}")

        for i, step in enumerate(self.steps, 1):
            try:
                tool_input = step["input_template"].format(**ctx)
            except KeyError as e:
                return f"❌ 工具链执行失败：模板变量 {e} 未定义"

            print(f"  步骤 {i} [{step['tool_name']}]: 输入前50字 = {tool_input[:50]}")
            result = registry.execute_tool(step["tool_name"], tool_input)
            ctx[step["output_key"]] = result
            print(f"  ✅ 步骤 {i} 完成，输出 {len(result)} 字符")

        final = ctx[self.steps[-1]["output_key"]]
        print(f"🎉 工具链 '{self.name}' 执行完成")
        return final


class ToolChainManager:
    def __init__(self, registry: ToolRegistry):
        self.registry = registry
        self.chains: Dict[str, ToolChain] = {}

    def register_chain(self, chain: ToolChain):
        self.chains[chain.name] = chain
        print(f"✅ 工具链 '{chain.name}' 已注册")

    def execute_chain(self, chain_name: str, input_data: str,
                      context: Optional[Dict[str, Any]] = None) -> str:
        if chain_name not in self.chains:
            return f"❌ 工具链 '{chain_name}' 不存在"
        return self.chains[chain_name].execute(self.registry, input_data, context)

    def list_chains(self) -> List[str]:
        return list(self.chains.keys())


# ── 演示：两步计算链 ──
print("=== 工具链演示 ===")

# 复用前面定义的 my_calculate 和 ToolRegistry
demo_registry = ToolRegistry()
demo_registry.register_function(
    name="my_calculator",
    description="数学计算",
    func=my_calculate  # 使用上方单元格中定义的函数
)
demo_registry.register_function(
    name="format_result",
    description="格式化结果",
    func=lambda x: f"计算结果是：{x}"
)

# 创建工具链：先计算，再格式化
chain = ToolChain(name="calc_then_format", description="计算后格式化输出")
chain.add_step("my_calculator",  "{input}",              "calc_result")
chain.add_step("format_result",  "{calc_result}",        "final_output")

manager = ToolChainManager(demo_registry)
manager.register_chain(chain)

print(f"\n已注册的工具链: {manager.list_chains()}")
output = manager.execute_chain("calc_then_format", "sqrt(16) + 2 * 3")
print(f"\n最终输出: {output}")

=== 工具链演示 ===


NameError: name 'my_calculate' is not defined

In [27]:
import asyncio
import concurrent.futures
from typing import Dict, List
from hello_agents import ToolRegistry

class AsyncToolExecutor:
    """异步工具执行器：并行执行多个工具任务，提升吞吐量"""

    def __init__(self, registry: ToolRegistry, max_workers: int = 4):
        self.registry = registry
        self.executor = concurrent.futures.ThreadPoolExecutor(max_workers=max_workers)

    async def execute_tool_async(self, tool_name: str, input_data: str) -> str:
        loop = asyncio.get_event_loop()
        return await loop.run_in_executor(
            self.executor,
            lambda: self.registry.execute_tool(tool_name, input_data)
        )

    async def execute_tools_parallel(self, tasks: List[Dict[str, str]]) -> List[str]:
        print(f"🚀 并行执行 {len(tasks)} 个任务")
        results = await asyncio.gather(*[
            self.execute_tool_async(t["tool_name"], t["input_data"])
            for t in tasks
        ])
        print("✅ 所有任务执行完成")
        return results

    def __del__(self):
        if hasattr(self, "executor"):
            self.executor.shutdown(wait=False)


# ── 演示：并行执行四个计算任务 ──
print("=== 异步并行工具执行演示 ===")

async_registry = ToolRegistry()
async_registry.register_function("calc", "数学计算", my_calculate)

tasks = [
    {"tool_name": "calc", "input_data": "2 + 2"},
    {"tool_name": "calc", "input_data": "sqrt(144)"},
    {"tool_name": "calc", "input_data": "7 * 8"},
    {"tool_name": "calc", "input_data": "100 / 4"},
]

async_executor = AsyncToolExecutor(async_registry)

import time
start = time.time()
results = await async_executor.execute_tools_parallel(tasks)  # Jupyter 支持顶层 await
elapsed = time.time() - start

print(f"\n耗时: {elapsed:.3f}s")
for task, result in zip(tasks, results):
    print(f"  {task['input_data']:<20} → {result}")

=== 异步并行工具执行演示 ===
✅ 工具 'calc' 已注册。
🚀 并行执行 4 个任务
✅ 所有任务执行完成

耗时: 0.003s
  2 + 2                → 4
  sqrt(144)            → 12.0
  7 * 8                → 56
  100 / 4              → 25.0


---
## 7.6 本章小结

恭喜你完成了第七章的全部内容！我们从零构建了 **HelloAgents** 框架的核心组件：

| 模块 | 文件 | 关键设计 |
|:--|:--|:--|
| LLM 接口 | `llm.py` | 多供应商 + 自动检测 + 本地模型 |
| 消息系统 | `message.py` | Pydantic 类型校验，对内丰富对外兼容 |
| 配置管理 | `config.py` | 环境变量覆盖，零配置可运行 |
| Agent 基类 | `agent.py` | ABC 抽象接口，强制实现 run() |
| SimpleAgent | `simple_agent.py` | 可选工具 + 流式响应 |
| ReActAgent | `react_agent.py` | Think → Act → Observe 循环 |
| ReflectionAgent | `reflection_agent.py` | 执行 → 反思 → 优化 |
| PlanAndSolveAgent | `plan_solve_agent.py` | 规划 → 逐步执行 |
| 工具系统 | `tools/` | 基类 + 注册表 + 工具链 + 异步执行 |

### 下一步
- 访问 [GitHub 完整示例](https://github.com/jjyaoao/HelloAgents/blob/main/examples/chapter07_basic_setup.py) 查看测试代码
- 第八章将在此基础上加入 **Memory 与 RAG 系统**，扩展 Agent 的长期记忆能力

---
### 📝 习题提示
1. 参考 7.2.1 节，尝试为 `HelloAgentsLLM` 添加 Gemini 或 Anthropic 的支持
2. 扩展 `MyReflectionAgent`，添加
机制：LLM 打分 ≥ 80 分时提前终止
3. 设计一个 `Tree-of-Thought Agent`，每步生成多条候选路径，选择最优的继续执行